<a href="https://colab.research.google.com/github/KruthikaVirupakshappa/DATA266-Lab1-Pair37/blob/jiayi-work/task1_llm/jiayi/src/Task1_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
SID4         =  4253 # last four digits of your SJSU ID
SEED         = SID4        # torch.manual_seed, np.random.seed, random_state
SLICE        = SID4 % 1000   # start index of your dataset subset
HP_ID        = SID4 % 6      # your assigned hyperparameter arm, Table 0.1
CLS_A, CLS_B = (SID4 % 10), ((SID4 // 10) % 10)   # your two focus classes

In [6]:
print(f"SID4: {SID4}, SEED: {SEED}, SLICE: {SLICE}, HP_ID: {HP_ID}, CLS_A, CLS_B: {CLS_A}, {CLS_B}")

SID4: 4253, SEED: 4253, SLICE: 253, HP_ID: 5, CLS_A, CLS_B: 3, 5


## Task 1
### data preparation

In [7]:
import random
import numpy as np
import torch
from pathlib import Path
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
import math
import torch.nn as nn
import torch.nn.functional as F
import time
import matplotlib.pyplot as plt

In [8]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [10]:
# load shared data from download_tinystories.py
from datasets import load_from_disk
from pathlib import Path
from datasets import load_dataset

dataset_folder = Path("task1_llm/data/tinystories")
data = load_from_disk(str(dataset_folder))

# inspect the original dataset
print(data)
print(data.keys())

# only use the original training set for training and validation
original_train = data["train"]

# use SEED to randomize original_train
shuffled_train = original_train.shuffle(seed=SEED)

# split the shuffled train
my_train = shuffled_train.select(range(100_000))
my_val = shuffled_train.select(range(100_000, 110000))

# inspect the splitted dataset by checking shape and a few data
print(len(my_train))
print(len(my_val))
print(my_train.column_names)
print(my_train[0])
print(my_val[0])

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})
dict_keys(['train', 'validation'])
100000
10000
['text']
{'text': 'Once upon a time, there was a little bird named Blue. Blue was very fast and loved to fly high in the sky. One day, Blue met a little girl named Lily. \n\n"Hello, Blue! You can fly very fast!" said Lily. \n\n"Yes, I can fly very fast!" replied Blue, happily. \n\nBut then, a big storm came and Blue got lost. Blue was scared and didn\'t know what to do. \n\n"Can you help me, Lily?" asked Blue. \n\n"Yes, I can help you!" said Lily. "Let\'s look for your history book. Maybe it will help us find your way home." \n\nSo, Lily and Blue searched for Blue\'s history book. They found it and read about all the places Blue had flown before. They followed the book and finally, Blue found his way home. \n\n"Thank you, Lily! You\'re my hero!" said Blue. \

In [11]:
train_text = "\n".join(my_train["text"])
val_text = "\n".join(my_val["text"])

chars = sorted(list(set(train_text)))
vocab_size = len(chars)

char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

def encode(s):
    return [char_to_idx[c] for c in s]

def decode(ids):
    return "".join(idx_to_char[i] for i in ids)

unseen_chars = set(val_text) - set(chars)
print("Unseen validation chars:", unseen_chars)

Unseen validation chars: set()


In [12]:
train_data = torch.tensor(encode(train_text), dtype=torch.long)
val_data = torch.tensor(encode(val_text), dtype=torch.long)

print("Vocab size:", vocab_size)
print("Train characters:", len(train_data))
print("Val characters:", len(val_data))

Vocab size: 152
Train characters: 89734384
Val characters: 8964594


In [13]:
class CharDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return (len(self.data) - 1) // self.seq_len

    def __getitem__(self, idx):
        start = idx * self.seq_len

        x = self.data[start : start + self.seq_len]
        y = self.data[start + 1 : start + self.seq_len + 1]

        return x, y

In [14]:
seq_len = 128
train_dataset = CharDataset(train_data, seq_len)
val_dataset = CharDataset(val_data, seq_len)

print(len(train_dataset))
print(len(val_dataset))

701049
70035


In [15]:
x, y = train_dataset[0]

print(x.shape)
print(y.shape)
print("Input:\n" + decode(x.tolist()))
print("Output:\n" + decode(y.tolist()))

torch.Size([128])
torch.Size([128])
Input:
Once upon a time, there was a little bird named Blue. Blue was very fast and loved to fly high in the sky. One day, Blue met a l
Output:
nce upon a time, there was a little bird named Blue. Blue was very fast and loved to fly high in the sky. One day, Blue met a li


In [16]:
generator = torch.Generator()
generator.manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, generator=generator)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

### GPT model

In [17]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "Embedding dimension must be 0 modulo number of heads."
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, C = x.shape

        QKV = self.qkv(x) # [B, T, 3*d_model]

        QKV = QKV.reshape(B, T, self.num_heads, 3*self.head_dim)
        QKV = QKV.permute(0, 2, 1, 3) # [Batch, Head, SeqLen, Dims]
        Q,K,V = QKV.chunk(3, dim=-1) # each [B, H, T, D]

        # Attention scores between every pair of token
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.head_dim)

        # causal mask
        # prevent token from attending future tokens
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float('-inf'))

        attn = F.softmax(scores, dim=-1)
        out = attn @ V
        # merge all attention heads back to [B, T, C]
        out = out.permute(0,2,1,3) # [Batch, SeqLen, Head, Dims]
        out = out.reshape(B,T,C)
        out = self.out_proj(out)


        return out


In [18]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, num_heads)
        self.ln2 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, 4*d_model),
            nn.GELU(),
            nn.Linear(4*d_model, d_model)
        )

    def forward(self, x):
        # self-attention with residual connection
        x = x + self.attn(self.ln1(x))
        # feedforward network with residual connection
        x = x + self.ff(self.ln2(x))
        return x

In [19]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, d_model=128, seq_len=128, num_heads=4):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_layers = 2
        self.seq_len = seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(seq_len, d_model)

        self.block1 = DecoderBlock(d_model, num_heads)
        self.block2 = DecoderBlock(d_model, num_heads)
        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, idx):
        B, T = idx.shape
        # Convert token IDs and positions into trainable embeddings
        tok = self.token_emb(idx)
        pos_ids = torch.arange(T, device=idx.device)
        pos = self.pos_emb(pos_ids)

        x = tok + pos
        # two stack decoder blocks
        x = self.block1(x)
        x = self.block2(x)
        x = self.ln(x)
        # project each token representation to vocabulary logits
        logits = self.head(x)

        return logits

model = TinyGPT(vocab_size, seq_len=seq_len).to(device)
print("Model parameters:", sum(p.numel() for p in model.parameters()))
print("Hidden dimension:", model.d_model)
print("Number of heads:", model.num_heads)
print("Number of layers:", model.num_layers)

Model parameters: 452248
Hidden dimension: 128
Number of heads: 4
Number of layers: 2


### Training

In [20]:
base_lr = 3e-4
optimizer = torch.optim.Adam(model.parameters(), lr=base_lr)
criterion = nn.CrossEntropyLoss()

epochs = 10
total_steps = epochs * len(train_loader)
# first 5% of steps for warmup
warmup_steps = int(0.05 * total_steps)

def lr_lambda(step):
    # linear warmup
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    # cosine decay after warmup
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    # decay from 1.0 to 0.1 of base learning rate
    return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda)

print("Total training steps:", total_steps)
print("Warmup steps:", warmup_steps)

Total training steps: 219080
Warmup steps: 10954


### smoke test

In [22]:
# raw log location
log_dir = Path("reproducibility/raw_logs")
log_dir.mkdir(parents=True, exist_ok=True)

log_path = log_dir / "task1_jiayi_smoke.log"

In [23]:
import copy

smoke_model = copy.deepcopy(model)

smoke_optimizer = torch.optim.Adam(smoke_model.parameters(), lr=base_lr)
smoke_scheduler = torch.optim.lr_scheduler.LambdaLR(smoke_optimizer, lr_lambda)

# Prevent accidentally overwriting an existing raw log
if log_path.exists():
    raise FileExistsError(
        f"{log_path} already exists. "
        "Use a new filename if you want to run another smoke test."
    )

In [24]:
with open(log_path, "w") as log_file:

    def log(message):
        print(message)
        log_file.write(message + "\n")
        log_file.flush()

    log("Task 1 GPT Smoke Test")
    log(f"Seed: {SEED}")
    log(f"Device: {device}")
    log(f"Base learning rate: {base_lr}")
    log("Train batches: 5")
    log("Validation batches: 2")

    smoke_model.train()

    for batch_idx, (x, y) in enumerate(train_loader):

        if batch_idx == 5:
            break

        x, y = x.to(device), y.to(device)

        logits = smoke_model(x)

        loss = criterion(
            logits.view(-1, vocab_size),
            y.view(-1)
        )

        assert torch.isfinite(loss), "Non-finite training loss"

        smoke_optimizer.zero_grad()
        loss.backward()
        smoke_optimizer.step()
        smoke_scheduler.step()

        current_lr = smoke_optimizer.param_groups[0]["lr"]

        log(
            f"Train batch {batch_idx + 1}: "
            f"loss={loss.item():.4f}, "
            f"lr={current_lr:.8f}"
        )


    smoke_model.eval()

    with torch.no_grad():

        for batch_idx, (x, y) in enumerate(val_loader):

            if batch_idx == 2:
                break

            x, y = x.to(device), y.to(device)

            logits = smoke_model(x)

            loss = criterion(
                logits.view(-1, vocab_size),
                y.view(-1)
            )

            assert torch.isfinite(loss), "Non-finite validation loss"

            log(
                f"Validation batch {batch_idx + 1}: "
                f"loss={loss.item():.4f}"
            )

    log("Smoke test passed.")

Task 1 GPT Smoke Test
Seed: 4253
Device: cuda
Base learning rate: 0.0003
Train batches: 5
Validation batches: 2
Train batch 1: loss=5.1553, lr=0.00000005
Train batch 2: loss=5.1757, lr=0.00000008
Train batch 3: loss=5.1835, lr=0.00000011
Train batch 4: loss=5.1608, lr=0.00000014
Train batch 5: loss=5.1727, lr=0.00000016
Validation batch 1: loss=5.1682
Validation batch 2: loss=5.1825
Smoke test passed.


### epoch = 10 training

In [ ]:
train_losses = []
val_losses = []

for ep in range(epochs):
    model.train()
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)
        # combine B and T for cross-entropy loss
        loss = criterion(logits.view(-1, vocab_size), y.view(-1))
        # backpropagation and parameter update
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        # loss for this epoch
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)

            logits = model(x)

            loss = criterion(
                logits.view(-1, vocab_size),
                y.view(-1)
            )

            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    print(f"Epoch {ep+1}/{epochs}, "f"Train Loss: {avg_train_loss:.4f}, ")
    print(f"Val Loss: {avg_val_loss:.4f}")

plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()

In [ ]:
@torch.no_grad()
def generate_temperature(prompt, max_new=100,  temperature=0.8):
    model.eval()
    ids = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new):
        ids_cond = ids[:, -model.seq_len:]
        logits = model(ids_cond)
        next_logits = logits[:, -1, :]
        # scale logits
        next_logits = next_logits / temperature
        # convert logits to probabilities and sample one token
        probs = F.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        ids = torch.cat([ids, next_token], dim=1)

    return decode(ids[0].tolist())

for temp in [0.5, 0.8, 1.0]:
    set_seed(SEED)
    print(f"\nTemperature = {temp}")
    print(generate_temperature("One day", 150, temperature=temp))
